# Quill — Notebook 02: Transforming Knots with Gates

In this notebook, we will apply **one X gate to one selected qubit** inside a Knot, then measure the Knot and decode the resulting character.

By the end, you should be able to explain:
- what an X gate does to a qubit;
- why changing one bit can change a character;
- how the measured bit string maps back to a character.

**Scope:** no superposition yet. We start with a basis state so the effect is easy to follow.

## 1. A Knot is a register, not a character-aware object

For this example, `Knot("H")` means an 8-qubit register prepared in the computational-basis state that encodes the ASCII value of `H`. The qubits do not know what a letter is; the character is our interpretation of the encoded bits.

ASCII values:

- `H` = decimal 72 = binary `01001000`
- `I` = decimal 73 = binary `01001001`

Only the final bit differs.

We write basis states in the usual left-to-right order, most-significant bit first:

$$|H\rangle_K = |01001000\rangle$$

$$|I\rangle_K = |01001001\rangle$$

The subscript `K` is Quill's label for a Knot. It is notation we define for this project, not standard quantum-mechanics notation.

## 2. What does the X gate do?

The Pauli-X gate is the quantum equivalent of a bit flip on a computational-basis state:

$$X = \begin{bmatrix}0 & 1\1 & 0\end{bmatrix}$$

$$X|0\rangle = |1\rangle, \qquad X|1\rangle = |0\rangle$$

For our character, we apply X to **qubit 0**, the least-significant bit (the rightmost bit in the binary string).

$$|01001000\rangle \xrightarrow{\,X_0\,} |01001001\rangle$$

So the encoded integer changes from 72 to 73. Decoding those values gives `H` and `I` respectively. The gate does not understand letters; it flips a qubit, and our encoding scheme tells us which character the resulting bits represent.

## 3. Prepare the Knot and inspect its state

We'll use Qiskit to prepare the basis state for `H`. Qiskit indexes qubit 0 as the least-significant bit, so bit index 0 corresponds to the rightmost bit of the displayed binary string.

In [ ]:
# If needed in Google Colab, uncomment and run:
# %pip -q install qiskit qiskit-aer

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector


def encode_knot(character: str) -> QuantumCircuit:
    """Prepare an 8-qubit Knot in the ASCII basis state for one character."""
    if len(character) != 1 or ord(character) > 127:
        raise ValueError("Use one ASCII character.")

    value = ord(character)
    circuit = QuantumCircuit(8)
    for bit in range(8):
        if (value >> bit) & 1:
            circuit.x(bit)
    return circuit

knot_h = encode_knot("H")
print("ASCII:", ord("H"))
print("Binary:", format(ord("H"), "08b"))
display(knot_h.draw("mpl"))

The circuit prepares a single basis state. In the conventional displayed bit order, its state is:

$$|\psi_H\rangle = |01001000\rangle$$

A state vector has one amplitude equal to 1 for this basis state and 0 for all other basis states. We do not need to print all 256 amplitudes to understand this example.

In [ ]:
state_h = Statevector.from_instruction(knot_h)
print("Nonzero basis-state amplitude:", state_h.data[72])
print("Probability of H's encoded state:", abs(state_h.data[72])**2)

## 4. Apply X to qubit 0

Make a copy of the prepared Knot, then apply `X` to its least-significant bit.

In [ ]:
knot_i = knot_h.copy()
knot_i.x(0)

display(knot_i.draw("mpl"))

state_i = Statevector.from_instruction(knot_i)
print("Probability of basis state 73:", abs(state_i.data[73])**2)

The mathematical transformation is:

$$X_0|01001000\rangle = |01001001\rangle$$

The result is still a definite basis state. We changed one encoded bit, not the meaning of a word.

## 5. Measure the Knot and decode the character

Measurement reads the computational-basis bits. Because this circuit contains no superposition, repeated runs should return the same bit string.

In [ ]:
from qiskit_aer import AerSimulator

measured = knot_i.copy()
measured.measure_all()

simulator = AerSimulator()
result = simulator.run(measured, shots=100).result()
counts = result.get_counts()
print(counts)

# Decode the measured 8-bit string back to an ASCII character.
# measure_all displays the classical bit string in conventional MSB-first order.
bit_string = max(counts, key=counts.get)
value = int(bit_string, 2)
print("Measured bits:", bit_string)
print("Decoded integer:", value)
print("Decoded character:", chr(value))

## 6. A useful property: applying X twice undoes the flip

The X gate is its own inverse:

$$X^2 = I$$

Therefore, applying X twice to the same qubit returns the original basis state.

In [ ]:
twice = knot_h.copy()
twice.x(0)
twice.x(0)

state_twice = Statevector.from_instruction(twice)
print("Probability of original H state:", abs(state_twice.data[72])**2)
print("Probability of I state:", abs(state_twice.data[73])**2)

## 7. Your turn

Before running the cell, predict the result: what character do we get if we apply X to **qubit 3** of `Knot("H")`? Remember that qubit 0 is the rightmost bit.

$$|01001000\rangle \xrightarrow{\,X_3\,} \ ?$$

In [ ]:
# Try it: change the qubit index and inspect the decoded result.
trial = encode_knot("H")
trial.x(3)
trial.measure_all()

trial_counts = simulator.run(trial, shots=100).result().get_counts()
trial_bits = max(trial_counts, key=trial_counts.get)
print("Measured bits:", trial_bits)
print("Decoded character:", chr(int(trial_bits, 2)))

## Recap

- A **Knot** is an 8-qubit register encoding one ASCII character in this example.
- An **X gate** flips the computational-basis value of its target qubit.
- Changing a bit can change which character the bit string decodes to.
- Measurement reveals a classical bit string; our decoder interprets that string as a character.
- The gate acts on the quantum state, not on the semantic meaning of the letter.

**Next notebook:** Hadamard and rotation gates, where a Knot can enter superposition and measurement outcomes can become probabilistic.